The imports and data:

In [35]:
import pandas as pd
import numpy as np
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [36]:
#Assigning names to the sets
train = pd.read_csv('/content/drive/MyDrive/Kaggle/churn_train.csv')
test = pd.read_csv('/content/drive/MyDrive/Kaggle/churn_test.csv')
accounts = pd.read_csv('/content/drive/MyDrive/Kaggle/accounts.csv')
demographics = pd.read_csv('/content/drive/MyDrive/Kaggle/demographics.csv')
services = pd.read_csv('/content/drive/MyDrive/Kaggle/services.csv')
submission = pd.read_csv('/content/drive/MyDrive/Kaggle/sample_submission.csv')

Cleaning the data

In [37]:
# Looking at the data sets
display(train.head())
display(test.head())
display(accounts.head())
display(demographics.head())
display(services.head())
display(submission.head())

,CustomerNo,Churn
0,1201,0
1,1202,0
2,1205,1
3,1206,0
4,1207,0


,CustomerNo
0,1203
1,1204
2,1208
3,1220
4,1221


,CustomerNo,BaseCharges,DOC,TotalCharges,DOE,ElectronicBilling,ContractType,PaymentMethod
0,1201,478.50,5-1-2012,5920.00,10-Jan-11,No,One year,Electronic check
1,1202,103.25,5-1-2012,103.25,6-Dec-11,No,Month-to-month,Electronic check
2,1203,399.75,5-1-2012,5217.00,10-Jan-11,Yes,Month-to-month,Bank transfer (automatic)
3,1204,324.75,5-1-2012,22757.50,6-Apr-06,No,One year,Bank transfer (automatic)
4,1205,372.50,5-1-2012,372.50,6-Dec-11,Yes,Month-to-month,Electronic check


,CustomerNo,Country,State,Retired,HasPartner,HasDependents,Education,Gender
0,1201,India,Maharashtra,0,2,2,Masters,Male
1,1202,India,Maharashtra,0,2,2,ProfessionalQalification,Male
2,1203,India,Maharashtra,0,2,2,ProfessionalQalification,Male
3,1204,India,Maharashtra,0,2,1,ProfessionalQalification,Male
4,1205,India,Maharashtra,0,2,2,ProfessionalQalification,Female


,CustomerNo,TypeOfService,SeviceDetails
0,1201,DeviceProtection,Yes
1,1201,HasPhoneService,1
2,1201,InternetServiceCategory,Fiber optic
3,1201,MultipleLines,No
4,1201,OnlineBackup,No


,CustomerNo,Churn
0,1203,0
1,1204,0
2,1208,0
3,1220,0
4,1221,0


In [38]:
# Converting services from long to wide format
services_wide = services.pivot_table(
    index='CustomerNo',
    columns='TypeOfService',
    values='SeviceDetails',
    aggfunc='first'
).reset_index()

# Viewing the new dataset
display(services_wide.head())


TypeOfService,CustomerNo,DeviceProtection,HasPhoneService,InternetServiceCategory,MultipleLines,OnlineBackup,OnlineSecurity,StreamingMovies,StreamingTelevision,TechnicalSupport
0,1201,Yes,1,Fiber optic,No,No,No,Yes,Yes,No
1,1202,No internet service,1,No,No,No internet service,No internet service,No internet service,No internet service,No internet service
2,1203,No,1,Fiber optic,No,No,No,Yes,No,No
3,1204,No,1,DSL,Yes,Yes,Yes,No,No,Yes
4,1205,No,1,Fiber optic,Yes,No,No,No,No,No


In [39]:
# Merging training data with customer information, They match well and will function better combined
train_full = train.merge(accounts, on='CustomerNo', how='left')

train_full = train_full.merge( demographics, on='CustomerNo', how='left')

train_full = train_full.merge(services_wide, on='CustomerNo', how='left')

In [40]:
# Merge testing data with customer information
test_full = test.merge(accounts, on='CustomerNo', how='left')

test_full = test_full.merge(
    demographics, on='CustomerNo', how='left'
)

test_full = test_full.merge(
    services_wide, on='CustomerNo', how='left'
)


In [41]:
#checking that everything combined well
print("Training data:", train_full.shape)
print("Testing data:", test_full.shape)

display(train_full.head())


Training data: (4238, 25)
Testing data: (1060, 24)


,CustomerNo,Churn,BaseCharges,DOC,TotalCharges,DOE,ElectronicBilling,ContractType,PaymentMethod,Country,...,Gender,DeviceProtection,HasPhoneService,InternetServiceCategory,MultipleLines,OnlineBackup,OnlineSecurity,StreamingMovies,StreamingTelevision,TechnicalSupport
0,1201,0,478.50,5-1-2012,5920.00,10-Jan-11,No,One year,Electronic check,India,...,Male,Yes,1,Fiber optic,No,No,No,Yes,Yes,No
1,1202,0,103.25,5-1-2012,103.25,6-Dec-11,No,Month-to-month,Electronic check,India,...,Male,No internet service,1,No,No,No internet service,No internet service,No internet service,No internet service,No internet service
2,1205,1,372.50,5-1-2012,372.50,6-Dec-11,Yes,Month-to-month,Electronic check,India,...,Female,No,1,Fiber optic,Yes,No,No,No,No,No
3,1206,0,99.50,5-1-2012,6775.50,7-Mar-06,Yes,Two year,Credit card (automatic),India,...,Male,No internet service,1,No,No,No internet service,No internet service,No internet service,No internet service,No internet service
4,1207,0,91.25,5-1-2012,2673.50,17-Sep-09,No,One year,Credit card (automatic),India,...,Male,No internet service,1,No,No,No internet service,No internet service,No internet service,No internet service,No internet service


In [42]:
# Getting a count customers in each churn category
print(train_full['Churn'].value_counts())

# Calculating percentages
print(train_full['Churn'].value_counts(normalize=True) * 100)

Churn
0    3189
1    1049
Name: count, dtype: int64
Churn
0    75.247758
1    24.752242
Name: proportion, dtype: float64


In [43]:
# Checking for missing values
print("Missing values:")
print(train_full.isnull().sum().sort_values(ascending=False))

# Checking data types
train_full.info()

Missing values:
Education                  9
TotalCharges               8
ContractType               3
Gender                     2
CustomerNo                 0
DOC                        0
DOE                        0
BaseCharges                0
Churn                      0
PaymentMethod              0
ElectronicBilling          0
Country                    0
State                      0
HasPartner                 0
Retired                    0
HasDependents              0
DeviceProtection           0
HasPhoneService            0
InternetServiceCategory    0
MultipleLines              0
OnlineBackup               0
OnlineSecurity             0
StreamingMovies            0
StreamingTelevision        0
TechnicalSupport           0
dtype: int64
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4238 entries, 0 to 4237
Data columns (total 25 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   CustomerNo               4

In [44]:
# Fixing missing numerical values with the median value
train_full['TotalCharges'] = train_full['TotalCharges'].fillna(
    train_full['TotalCharges'].median()
)

# Fill missing categorical values with the most common category
for column in ['Education', 'ContractType', 'Gender']:

    train_full[column] = train_full[column].fillna(
        train_full[column].mode()[0]
    )

# Checking missing missing values again to make sure i got everything
print(train_full.isnull().sum().sum())


0


In [45]:
print(train_full['Churn'].value_counts())

print("\nChurn percentages:")
print(train_full['Churn'].value_counts(normalize=True) * 100)

Churn
0    3189
1    1049
Name: count, dtype: int64

Churn percentages:
Churn
0    75.247758
1    24.752242
Name: proportion, dtype: float64


In [46]:
#Seeing whats useful to keep
print("DOC unique values:")
print(train_full['DOC'].nunique())

print("\nDOE unique values:")
print(train_full['DOE'].nunique())

print("\nSample dates:")
display(train_full[['DOC', 'DOE', 'Churn']].head(10))

DOC unique values:
1

DOE unique values:
73

Sample dates:


,DOC,DOE,Churn
0,5-1-2012,10-Jan-11,0
1,5-1-2012,6-Dec-11,0
2,5-1-2012,6-Dec-11,1
3,5-1-2012,7-Mar-06,0
4,5-1-2012,17-Sep-09,0
5,5-1-2012,10-Jan-11,0
6,5-1-2012,23-Aug-08,0
7,5-1-2012,11-Mar-11,0
8,5-1-2012,12-Oct-10,0
9,5-1-2012,7-Oct-11,1


In [47]:
# Convert DOE into datetime format
train_full['DOE'] = pd.to_datetime(
    train_full['DOE'],
    format='%d-%b-%y',
    errors='coerce'
)

test_full['DOE'] = pd.to_datetime(
    test_full['DOE'],
    format='%d-%b-%y',
    errors='coerce'
)

# Set the common observation date
reference_date = pd.Timestamp('2012-05-01')

# Calculate customer tenure in days
train_full['TenureDays'] = (
    reference_date - train_full['DOE']
).dt.days

test_full['TenureDays'] = (
    reference_date - test_full['DOE']
).dt.days

# Remove original date columns
train_full = train_full.drop(columns=['DOC', 'DOE'])
test_full = test_full.drop(columns=['DOC', 'DOE'])

# Check our new variable
display(train_full[['CustomerNo', 'TenureDays', 'Churn']].head(10))


,CustomerNo,TenureDays,Churn
0,1201,477,0
1,1202,147,0
2,1205,147,1
3,1206,2247,0
4,1207,957,0
5,1209,477,0
6,1210,1347,0
7,1211,417,0
8,1212,567,0
9,1213,207,1


In [48]:
# Calculate average tenure for each churn group
print(
    train_full.groupby('Churn')['TenureDays'].mean()
)

Churn
0    1274.836312
1     679.192564
Name: TenureDays, dtype: float64


In [49]:

# Identify numerical and categorical columns
numeric_cols = train_full.select_dtypes(
    include=['number']
).columns.drop(['CustomerNo', 'Churn'])

categorical_cols = train_full.select_dtypes(
    include=['object']
).columns

# Fill missing numerical values
for col in numeric_cols:
    test_full[col] = test_full[col].fillna(
        train_full[col].median()
    )

# Fill missing categorical values
for col in categorical_cols:
    test_full[col] = test_full[col].fillna(
        train_full[col].mode()[0]
    )


In [50]:

# Remove Churn and CustomerNo from predictors
X = train_full.drop(
    columns=['Churn', 'CustomerNo']
)

# Churn is our target
y = train_full['Churn']

# Prepare the Kaggle test data
X_test = test_full.drop(columns=['CustomerNo'])


In [51]:
# Combine training and testing predictors temporarily
combined = pd.concat([X, X_test], axis=0)

# Convert categorical variables into numerical columns
combined_encoded = pd.get_dummies(combined, drop_first=True)

# Separate training and testing data again
X_encoded = combined_encoded.iloc[:len(X)]
X_test_encoded = combined_encoded.iloc[len(X):]

print("Training predictors:", X_encoded.shape)
print("Testing predictors:", X_test_encoded.shape)

Training predictors: (4238, 36)
Testing predictors: (1060, 36)


In [52]:
from sklearn.model_selection import train_test_split

X_train, X_val, y_train, y_val = train_test_split(
    X_encoded,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)


In [53]:

from sklearn.model_selection import train_test_split

X_train, X_val, y_train, y_val = train_test_split(
    X_encoded,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)


The Logistic Regresion Model:

In [54]:

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    roc_auc_score
)

# Build the Logistic Regression model
log_model = LogisticRegression(max_iter=1000)

# Train the model
log_model.fit(X_train, y_train)

# Make predictions
y_pred = log_model.predict(X_val)

# Evaluate the model
print("Accuracy:", accuracy_score(y_val, y_pred))

print("\nClassification Report:")
print(classification_report(y_val, y_pred))

# Calculate ROC-AUC
y_prob = log_model.predict_proba(X_val)[:, 1]

print("ROC-AUC:", roc_auc_score(y_val, y_prob))


Accuracy: 0.8337264150943396

Classification Report:
              precision    recall  f1-score   support

           0       0.86      0.93      0.89       638
           1       0.72      0.54      0.62       210

    accuracy                           0.83       848
   macro avg       0.79      0.74      0.76       848
weighted avg       0.83      0.83      0.83       848

ROC-AUC: 0.8713464696223318


/usr/local/lib/python3.13/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


Preparing for submission

In [55]:

from sklearn.linear_model import LogisticRegression

# 1. Re-fit Logistic Regression on 100% of training data
log_model_full = LogisticRegression(max_iter=1000, random_state=42)
log_model_full.fit(X_encoded, y)

# 2. Predict churn on the Kaggle test set
test_preds = log_model_full.predict(X_test_encoded)

# 3. Populate submission dataframe and export CSV
submission['Churn'] = test_preds

submission.to_csv('submission.csv', index=False)

print("Logistic Regression submission file 'submission.csv' generated successfully!")

Logistic Regression submission file 'submission.csv' generated successfully!


/usr/local/lib/python3.13/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [56]:
import os

# Set destination path to Google Drive
output_dir = '/content/drive/MyDrive/Kaggle'

In [57]:
# Save submission CSV to Google Drive
submission.to_csv(f'{output_dir}/submission.csv', index=False)

print(f"File successfully saved to Google Drive: {output_dir}/submission.csv")

File successfully saved to Google Drive: /content/drive/MyDrive/Kaggle/submission.csv
